# G11 — Taranis Black
## MVP de People Analytics | Rota Zero + Rota 3

Este notebook implementa uma análise reproduzível em camadas conceituais Bronze → Prata → Ouro. Os dados são sintéticos e devem ser carregados pelo usuário no Colab.

**Uso responsável:** os resultados são agregados; não usar o notebook para decisões automatizadas de RH.

In [1]:
!pip -q install openpyxl pandas matplotlib seaborn

import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set_theme(style="whitegrid")
print("Ambiente pronto.")

Ambiente pronto.


In [2]:
uploaded = files.upload()
filename = next(iter(uploaded))
print("Arquivo carregado:", filename)

Arquivo carregado: 04_Base_Colaboradores_People_Analytics.xlsx


In [3]:
# Bronze: leitura das fontes originais
stream = io.BytesIO(uploaded[filename])
xl = pd.ExcelFile(stream)
print("Abas:", xl.sheet_names)
base_bronze = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name="Base_Colaboradores")
clima_bronze = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name="Pesquisa_Clima_Qualitativa")
kpis_bronze = pd.read_excel(io.BytesIO(uploaded[filename]), sheet_name="KPIs_Gerais_RH")

Abas: ['Base_Colaboradores', 'Pesquisa_Clima_Qualitativa', 'KPIs_Gerais_RH']


In [4]:
# Prata: limpeza, tipos, minimização (remoção de Nome e Salário) e validações
base = base_bronze.copy()
base = base[base["ID Colaborador"].ne("ID Colaborador")].copy()
colunas_sensiveis = [c for c in ["Nome", "Salário Mensal (R$)"] if c in base.columns]
base = base.drop(columns=colunas_sensiveis)
base["Status"] = base["Status"].astype(str).str.strip()
base["Departamento"] = base["Departamento"].astype(str).str.strip()
for col in ["Nota eNPS (0-10)", "Absenteísmo (Dias no Ano)", "Tempo de Casa (Meses)"]:
    base[col] = pd.to_numeric(base[col], errors="coerce")
base["Dias até Desligamento"] = pd.to_numeric(base["Dias até Desligamento"], errors="coerce")
base["Desligado_90d"] = base["Status"].eq("Desligado") & base["Dias até Desligamento"].le(90)

clima = clima_bronze[clima_bronze["ID Registro"].ne("ID Registro")].copy()
for col in ["Nota Liderança (1-5)", "Nota Ferramentas / Processos (1-5)", "Nota Carga de Trabalho (1-5)"]:
    clima[col] = pd.to_numeric(clima[col], errors="coerce")

print("Colunas sensíveis removidas na Prata:", colunas_sensiveis)
print("Registros base:", len(base))
print("Registros clima:", len(clima))
print("Nulos nas colunas críticas:")
print(base[["Status", "Departamento", "Nota eNPS (0-10)"]].isna().sum())

Colunas sensíveis removidas na Prata: ['Nome', 'Salário Mensal (R$)']
Registros base: 280
Registros clima: 14
Nulos nas colunas críticas:
Status              0
Departamento        0
Nota eNPS (0-10)    0
dtype: int64


In [5]:
# Ouro: indicadores executivos
resumo = (
    base.groupby("Departamento", dropna=False)
    .agg(
        colaboradores=("ID Colaborador", "count"),
        desligamentos=("Status", lambda s: (s == "Desligado").sum()),
        enps_medio=("Nota eNPS (0-10)", "mean"),
        absenteismo_medio=("Absenteísmo (Dias no Ano)", "mean"),
        desligados_90d=("Desligado_90d", "sum"),
    )
    .reset_index()
)
resumo["turnover_observado_pct"] = 100 * resumo["desligamentos"] / resumo["colaboradores"]
resumo["turnover_90d_sobre_base_pct"] = 100 * resumo["desligados_90d"] / resumo["colaboradores"]
resumo = resumo.sort_values("turnover_observado_pct", ascending=False)
resumo.round(2)

            Departamento  colaboradores  desligamentos  enps_medio  absenteismo_medio  desligados_90d  turnover_observado_pct  turnover_90d_sobre_base_pct
5     Suporte ao Cliente             65             25        5.51               8.85               3                   38.46                         4.62
0          Comercial B2B             55             17        5.58               7.16               5                   30.91                         9.09
4              Marketing             25              4        6.16               6.20               1                   16.00                         4.00
6  Operações e Logística             60              9        6.22               6.32               1                   15.00                         1.67
2               Finanças             30              4        6.27               5.73               1                   13.33                         3.33
3        Gente e Cultura             15              2        7.53    

In [6]:
# Indicadores gerais e verificação de coerência
total = {
    "colaboradores": len(base),
    "ativos": int((base["Status"] == "Ativo").sum()),
    "desligados": int((base["Status"] == "Desligado").sum()),
    "turnover_observado_pct": round(100 * (base["Status"] == "Desligado").mean(), 2),
    "desligados_90d": int(base["Desligado_90d"].sum()),
    "desligados_90d_sobre_base_pct": round(100 * base["Desligado_90d"].mean(), 2),
    "enps_medio": round(base["Nota eNPS (0-10)"].mean(), 2),
}
pd.Series(total, name="valor")

colaboradores                    280.00
ativos                           216.00
desligados                        64.00
turnover_observado_pct            22.86
desligados_90d                    13.00
desligados_90d_sobre_base_pct      4.64
enps_medio                         6.08
Name: valor, dtype: float64

In [7]:
# Clima qualitativo agregado
clima_resumo = clima[[
    "Nota Liderança (1-5)",
    "Nota Ferramentas / Processos (1-5)",
    "Nota Carga de Trabalho (1-5)",
]].mean().round(2)
clima_resumo

Nota Liderança (1-5)                  3.07
Nota Ferramentas / Processos (1-5)    2.43
Nota Carga de Trabalho (1-5)          2.07
dtype: float64

In [8]:
# Visual 1: turnover observado por departamento
plt.figure(figsize=(10, 5))
sns.barplot(data=resumo, x="turnover_observado_pct", y="Departamento", color="#2563eb")
plt.xlabel("Turnover observado (%)")
plt.ylabel("Departamento")
plt.title("Turnover observado por departamento")
plt.xlim(0, max(45, resumo["turnover_observado_pct"].max() + 5))
plt.tight_layout()
plt.show()

In [9]:
# Visual 2: dimensões da pesquisa de clima
plt.figure(figsize=(8, 4))
ax = clima_resumo.sort_values().plot(kind="barh", color=["#f59e0b", "#8b5cf6", "#10b981"])
ax.set_xlim(0, 5)
ax.set_xlabel("Média (1 a 5)")
ax.set_title("Pesquisa de clima: médias agregadas")
plt.tight_layout()
plt.show()

## Hipóteses para o AI Canvas

1. Notas mais baixas de liderança podem estar associadas a maior risco de turnover.
2. Dificuldades em ferramentas/processos e carga de trabalho podem afetar a experiência das pessoas e a eficiência operacional.
3. O período inicial de até 90 dias merece uma intervenção de onboarding e liderança.
4. O simulador SBI é um caso de uso de impacto potencialmente alto e esforço médio.

Estas são hipóteses exploratórias. O notebook não demonstra causalidade.

In [10]:
# Matriz de priorização qualitativa do MVP
priorizacao = pd.DataFrame({
    "caso_uso": ["Simulador SBI", "Análise assistida de clima", "Onboarding inteligente"],
    "impacto": ["Alto", "Médio/alto", "Alto"],
    "esforco": ["Médio", "Baixo/médio", "Alto"],
    "prioridade": [1, 2, 3],
    "motivo": [
        "Missão obrigatória da Rota 3 e demonstrável",
        "Aproveita a base e fortalece o diagnóstico",
        "Exige jornada e governança mais amplas",
    ],
})
priorizacao

                     caso_uso     impacto      esforco  prioridade                                       motivo
0               Simulador SBI        Alto        Médio           1  Missão obrigatória da Rota 3 e demonstrável
1  Análise assistida de clima  Médio/alto  Baixo/médio           2   Aproveita a base e fortalece o diagnóstico
2      Onboarding inteligente        Alto         Alto           3       Exige jornada e governança mais amplas

## Próximo passo

Usar os números e gráficos revisados no Google Docs oficial. O simulador SBI deve ser apresentado como ferramenta de treinamento com supervisão humana; não deve ser usado para contratar, promover, desligar ou punir pessoas.